# Project · A convolutional network from scratch

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). Write every piece of a small CNN in NumPy, on the 8×8 handwritten digits: convolution (via im2col), ReLU, max pooling, a dense layer and softmax, forward and backward. Gradient-check it, train it, and compare it with an MLP of about the same size. Then shift the test images by one pixel to see what weight sharing buys you. Uses Murphy ch. 13–14 and D2L ch. 7.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · Data as images

Load the digits as (N, 1, 8, 8) arrays scaled to [0, 1], split 80/20 with a fixed seed, and one-hot the labels.

In [ ]:
# Your turn
import numpy as np
from sklearn.datasets import load_digits
dg = load_digits(); Xall = dg.images[:, None] / 16.0; yall = dg.target
rng = np.random.default_rng(0); idx = rng.permutation(len(yall)); cut = int(0.8 * len(yall))
Xtr, Xte, ytr, yte = Xall[idx[:cut]], Xall[idx[cut:]], yall[idx[:cut]], yall[idx[cut:]]
Ytr = ...  # TODO: one-hot labels
print(Xtr.shape, Xte.shape)

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
from sklearn.datasets import load_digits
dg = load_digits(); Xall = dg.images[:, None] / 16.0; yall = dg.target
rng = np.random.default_rng(0); idx = rng.permutation(len(yall)); cut = int(0.8 * len(yall))
Xtr, Xte, ytr, yte = Xall[idx[:cut]], Xall[idx[cut:]], yall[idx[:cut]], yall[idx[cut:]]
Ytr = np.eye(10)[ytr]  # one-hot labels
print(Xtr.shape, Xte.shape)

<details><summary><b>Answer and explanation</b></summary>

1,437 training and 360 test images, each 1 channel × 8 × 8. Keeping the image shape (instead of a flat 64-vector) is the whole point: convolution needs to know which pixels are neighbours.

Expected output:
```
(1437, 1, 8, 8) (360, 1, 8, 8)
```

</details>

## Milestone 2 · Convolution with im2col

A 3×3 convolution with padding 1 is a matrix multiply in disguise: copy every 3×3 patch into a row (im2col), then multiply by the (9·C_in × C_out) kernel matrix. Implement it and check it against a plain four-loop version.

In [ ]:
# Your turn
def im2col(X, k=3, pad=1):
    N, C, H, W = X.shape; Xp = np.pad(X, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    cols = np.empty((N, H, W, C, k, k))
    for i in range(k):
        for j in range(k):
            cols[:, :, :, :, i, j] = ...  # TODO: every patch becomes a row
    return cols.reshape(N * H * W, C * k * k)
def conv_forward(X, K, b):                                # K: (C_out, C_in, 3, 3)
    N, C, H, W = X.shape; cols = im2col(X)
    out = cols @ K.reshape(len(K), -1).T + b
    return out.reshape(N, H, W, len(K)).transpose(0, 3, 1, 2), cols
def conv_naive(X, K, b):
    N, C, H, W = X.shape; Xp = np.pad(X, ((0, 0), (0, 0), (1, 1), (1, 1))); out = np.zeros((N, len(K), H, W))
    for n in range(N):
        for f in range(len(K)):
            for i in range(H):
                for j in range(W): out[n, f, i, j] = np.sum(Xp[n, :, i:i + 3, j:j + 3] * K[f]) + b[f]
    return out
K = rng.normal(0, 0.3, (8, 1, 3, 3)); bK = np.zeros(8)
fast = conv_forward(Xtr[:5], K, bK)[0]
print("output shape", fast.shape, "  max difference vs loops:", np.abs(fast - conv_naive(Xtr[:5], K, bK)).max())

In [ ]:
# @title Solution 2 (double-click to show the code)
def im2col(X, k=3, pad=1):
    N, C, H, W = X.shape; Xp = np.pad(X, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    cols = np.empty((N, H, W, C, k, k))
    for i in range(k):
        for j in range(k):
            cols[:, :, :, :, i, j] = Xp[:, :, i:i + H, j:j + W].transpose(0, 2, 3, 1)  # every patch becomes a row
    return cols.reshape(N * H * W, C * k * k)
def conv_forward(X, K, b):                                # K: (C_out, C_in, 3, 3)
    N, C, H, W = X.shape; cols = im2col(X)
    out = cols @ K.reshape(len(K), -1).T + b
    return out.reshape(N, H, W, len(K)).transpose(0, 3, 1, 2), cols
def conv_naive(X, K, b):
    N, C, H, W = X.shape; Xp = np.pad(X, ((0, 0), (0, 0), (1, 1), (1, 1))); out = np.zeros((N, len(K), H, W))
    for n in range(N):
        for f in range(len(K)):
            for i in range(H):
                for j in range(W): out[n, f, i, j] = np.sum(Xp[n, :, i:i + 3, j:j + 3] * K[f]) + b[f]
    return out
K = rng.normal(0, 0.3, (8, 1, 3, 3)); bK = np.zeros(8)
fast = conv_forward(Xtr[:5], K, bK)[0]
print("output shape", fast.shape, "  max difference vs loops:", np.abs(fast - conv_naive(Xtr[:5], K, bK)).max())

<details><summary><b>Answer and explanation</b></summary>

The two versions agree to 2.5e−16 (floating-point noise). im2col turns 'slide a 3×3 window everywhere' into one big matrix multiply: (N·64 patches × 9 values) @ (9 × 8 filters). That's how real libraries make convolution fast on GPUs. The weight sharing is visible in the shapes: the same 9 numbers per filter are used at all 64 positions (Murphy §14.2.1, D2L §7.2).

Expected output:
```
output shape (5, 8, 8, 8)   max difference vs loops: 2.498001805406602e-16
```

</details>

## Milestone 3 · Pool, flatten, classify: the forward pass

Chain conv (8 filters) → ReLU → 2×2 max pool → flatten (8·4·4 = 128) → dense to 10 → softmax. Write a forward function that also keeps whatever the backward pass will need. Count the parameters and check that the initial loss is about log 10 ≈ 2.30.

In [ ]:
# Your turn
def init(seed=0):
    r = np.random.default_rng(seed)
    return {"K": r.normal(0, np.sqrt(2 / 9), (8, 1, 3, 3)), "bK": np.full(8, 0.01), "W": r.normal(0, np.sqrt(1 / 128), (128, 10)) * 0.1, "b": np.zeros(10)}
def forward(P, X):
    z, cols = conv_forward(X, P["K"], P["bK"]); a = np.maximum(z, 0)
    N = len(X); win = a.reshape(N, 8, 4, 2, 4, 2).transpose(0, 1, 2, 4, 3, 5).reshape(N, 8, 4, 4, 4)
    arg = win.argmax(-1); pooled = ...  # TODO: 2x2 max pool
    mask = (np.arange(4) == arg[..., None])                               # one winner per window (ties -> first)
    mask = mask.reshape(N, 8, 4, 4, 2, 2).transpose(0, 1, 2, 4, 3, 5)      # back to (N, 8, 4, 2, 4, 2) layout
    h = pooled.reshape(N, -1); s = h @ P["W"] + P["b"]
    s -= s.max(1, keepdims=True); p = np.exp(s); p /= p.sum(1, keepdims=True)
    return p, (cols, z, mask, h)
P = init(); p, _ = forward(P, Xtr[:200])
print("parameters:", sum(v.size for v in P.values()), "  initial loss", -np.mean(np.log(p[np.arange(200), ytr[:200]])).round(3))

In [ ]:
# @title Solution 3 (double-click to show the code)
def init(seed=0):
    r = np.random.default_rng(seed)
    return {"K": r.normal(0, np.sqrt(2 / 9), (8, 1, 3, 3)), "bK": np.full(8, 0.01), "W": r.normal(0, np.sqrt(1 / 128), (128, 10)) * 0.1, "b": np.zeros(10)}
def forward(P, X):
    z, cols = conv_forward(X, P["K"], P["bK"]); a = np.maximum(z, 0)
    N = len(X); win = a.reshape(N, 8, 4, 2, 4, 2).transpose(0, 1, 2, 4, 3, 5).reshape(N, 8, 4, 4, 4)
    arg = win.argmax(-1); pooled = np.take_along_axis(win, arg[..., None], -1)[..., 0]  # 2x2 max pool
    mask = (np.arange(4) == arg[..., None])                               # one winner per window (ties -> first)
    mask = mask.reshape(N, 8, 4, 4, 2, 2).transpose(0, 1, 2, 4, 3, 5)      # back to (N, 8, 4, 2, 4, 2) layout
    h = pooled.reshape(N, -1); s = h @ P["W"] + P["b"]
    s -= s.max(1, keepdims=True); p = np.exp(s); p /= p.sum(1, keepdims=True)
    return p, (cols, z, mask, h)
P = init(); p, _ = forward(P, Xtr[:200])
print("parameters:", sum(v.size for v in P.values()), "  initial loss", -np.mean(np.log(p[np.arange(200), ytr[:200]])).round(3))

<details><summary><b>Answer and explanation</b></summary>

1,370 parameters: 72 kernel weights + 8 conv biases + 1,280 dense weights + 10 biases. The initial loss is 2.302 = log 10: with tiny output weights every class gets probability ≈ 1/10, exactly as it should before learning. If your initial loss is far from log 10, something is wrong with the initialisation or the softmax.

Expected output:
```
parameters: 1370   initial loss 2.302
```

</details>

## Milestone 4 · Backward pass and a gradient check

Send the blame back: softmax+cross-entropy gives (p − y)/N, then dense, then un-pool (the gradient goes only to the max position), then ReLU (zero where z ≤ 0), then the conv weights (colsᵀ · gradient). Check a few weights of each parameter against finite differences.

In [ ]:
# Your turn
def backward(P, X, Y, cache, p):
    cols, z, mask, h = cache; N = len(X)
    ds = (p - Y) / N
    g = {"W": h.T @ ds, "b": ds.sum(0)}
    dh = (ds @ P["W"].T).reshape(N, 8, 4, 1, 4, 1)
    da = ...  # TODO: un-pool: blame goes to the max only
    dz = ...  # TODO: through ReLU
    dzc = dz.transpose(0, 2, 3, 1).reshape(-1, 8)
    g["K"] = (dzc.T @ cols).reshape(P["K"].shape); g["bK"] = ...  # TODO: conv weights: patches^T x blame
    return g
def loss(P, X, y): return -np.mean(np.log(forward(P, X)[0][np.arange(len(y)), y]))
Xb, yb = Xtr[:64], ytr[:64]; p, cache = forward(P, Xb); g = backward(P, Xb, np.eye(10)[yb], cache, p)
r = np.random.default_rng(5)
for k in ["K", "bK", "W", "b"]:
    i = tuple(int(r.integers(0, s)) for s in P[k].shape); h = 1e-5
    P[k][i] += h; lp = loss(P, Xb, yb); P[k][i] -= 2 * h; lm = loss(P, Xb, yb); P[k][i] += h
    num = (lp - lm) / (2 * h)
    print(f"{k:>2}{list(i)}: analytic {g[k][i]: .6f}  numeric {num: .6f}")

In [ ]:
# @title Solution 4 (double-click to show the code)
def backward(P, X, Y, cache, p):
    cols, z, mask, h = cache; N = len(X)
    ds = (p - Y) / N
    g = {"W": h.T @ ds, "b": ds.sum(0)}
    dh = (ds @ P["W"].T).reshape(N, 8, 4, 1, 4, 1)
    da = (mask * dh).reshape(N, 8, 8, 8)  # un-pool: blame goes to the max only
    dz = da * (z > 0)  # through ReLU
    dzc = dz.transpose(0, 2, 3, 1).reshape(-1, 8)
    g["K"] = (dzc.T @ cols).reshape(P["K"].shape); g["bK"] = dzc.sum(0)  # conv weights: patches^T x blame
    return g
def loss(P, X, y): return -np.mean(np.log(forward(P, X)[0][np.arange(len(y)), y]))
Xb, yb = Xtr[:64], ytr[:64]; p, cache = forward(P, Xb); g = backward(P, Xb, np.eye(10)[yb], cache, p)
r = np.random.default_rng(5)
for k in ["K", "bK", "W", "b"]:
    i = tuple(int(r.integers(0, s)) for s in P[k].shape); h = 1e-5
    P[k][i] += h; lp = loss(P, Xb, yb); P[k][i] -= 2 * h; lm = loss(P, Xb, yb); P[k][i] += h
    num = (lp - lm) / (2 * h)
    print(f"{k:>2}{list(i)}: analytic {g[k][i]: .6f}  numeric {num: .6f}")

<details><summary><b>Answer and explanation</b></summary>

All four agree to 6 digits, so the backward pass is right. Two details matter. (1) Max pooling passes blame only to the winning input of each window, with exactly one winner even when several values tie. Sending it to every tied value double-counts, and the gradient check catches that. (2) The conv bias starts at 0.01 rather than 0, so blank background pixels don't sit exactly on ReLU's kink, where the derivative is undefined and finite differences disagree with any choice you make. The conv weight gradient is colsᵀ · blame: the same im2col trick, run backwards (Murphy §13.3).

Expected output:
```
 K[5, 0, 2, 0]: analytic -0.000296  numeric -0.000296
bK[6]: analytic -0.000364  numeric -0.000364
 W[60, 5]: analytic -0.000788  numeric -0.000788
 b[6]: analytic -0.037585  numeric -0.037585
```

</details>

## Milestone 5 · Train it

Train with Adam (learning rate 0.01), mini-batches of 64, for 15 epochs. Print test accuracy every 5 epochs.

In [ ]:
# Your turn
def train(P, epochs=15, lr=0.01, seed=1):
    r = np.random.default_rng(seed); m = {k: np.zeros_like(v) for k, v in P.items()}; v2 = {k: np.zeros_like(v) for k, v in P.items()}; t = 0
    for ep in range(1, epochs + 1):
        for bi in np.array_split(r.permutation(len(Xtr)), len(Xtr) // 64):
            p, cache = forward(P, Xtr[bi]); g = backward(P, Xtr[bi], Ytr[bi], cache, p); t += 1
            for k in P:
                m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
                P[k] -= ...  # TODO: Adam step
        if ep % 5 == 0: print(f"epoch {ep:>2}: test accuracy {np.mean(forward(P, Xte)[0].argmax(1) == yte):.4f}")
    return P
P = train(init())

In [ ]:
# @title Solution 5 (double-click to show the code)
def train(P, epochs=15, lr=0.01, seed=1):
    r = np.random.default_rng(seed); m = {k: np.zeros_like(v) for k, v in P.items()}; v2 = {k: np.zeros_like(v) for k, v in P.items()}; t = 0
    for ep in range(1, epochs + 1):
        for bi in np.array_split(r.permutation(len(Xtr)), len(Xtr) // 64):
            p, cache = forward(P, Xtr[bi]); g = backward(P, Xtr[bi], Ytr[bi], cache, p); t += 1
            for k in P:
                m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
                P[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)  # Adam step
        if ep % 5 == 0: print(f"epoch {ep:>2}: test accuracy {np.mean(forward(P, Xte)[0].argmax(1) == yte):.4f}")
    return P
P = train(init())

<details><summary><b>Answer and explanation</b></summary>

Test accuracy reaches ≈ 97.5% after 15 epochs (95.0% at epoch 5, 96.4% at epoch 10), with 1,370 parameters and code you wrote entirely yourself. The whole training run takes under a second because im2col turns the work into a few matrix multiplies.

Expected output:
```
epoch  5: test accuracy 0.9500
epoch 10: test accuracy 0.9639
epoch 15: test accuracy 0.9750
```

</details>

## Milestone 6 · Same budget, no convolution

Build an MLP with a similar parameter count (64 → 20 ReLU → 10: 1,510 parameters vs the CNN's 1,370) and train it the same way. Compare test accuracy.

In [ ]:
# Your turn
def mlp_init(seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, np.sqrt(2 / 64), (64, 20)), "b1": np.zeros(20), "W2": r.normal(0, np.sqrt(1 / 20), (20, 10)), "b2": np.zeros(10)}
def mlp_forward(Q, X):
    x = X.reshape(len(X), -1); z = x @ Q["W1"] + Q["b1"]; h = np.maximum(z, 0); s = h @ Q["W2"] + Q["b2"]
    s -= s.max(1, keepdims=True); p = np.exp(s); p /= p.sum(1, keepdims=True); return p, (x, z, h)
def mlp_backward(Q, Y, cache, p):
    x, z, h = cache; ds = (p - Y) / len(Y); dh = ...  # TODO: same chain rule, no patches
    return {"W2": h.T @ ds, "b2": ds.sum(0), "W1": x.T @ dh, "b1": dh.sum(0)}
Q = mlp_init(); r = np.random.default_rng(1); m = {k: 0 * v for k, v in Q.items()}; v2 = {k: 0 * v for k, v in Q.items()}; t = 0
for ep in range(15):
    for bi in np.array_split(r.permutation(len(Xtr)), len(Xtr) // 64):
        p, c = mlp_forward(Q, Xtr[bi]); g = mlp_backward(Q, Ytr[bi], c, p); t += 1
        for k in Q:
            m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
            Q[k] -= 0.01 * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)
print("MLP parameters:", sum(v.size for v in Q.values()))
print(f"CNN test accuracy {np.mean(forward(P, Xte)[0].argmax(1) == yte):.4f}   MLP test accuracy {np.mean(mlp_forward(Q, Xte)[0].argmax(1) == yte):.4f}")

In [ ]:
# @title Solution 6 (double-click to show the code)
def mlp_init(seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, np.sqrt(2 / 64), (64, 20)), "b1": np.zeros(20), "W2": r.normal(0, np.sqrt(1 / 20), (20, 10)), "b2": np.zeros(10)}
def mlp_forward(Q, X):
    x = X.reshape(len(X), -1); z = x @ Q["W1"] + Q["b1"]; h = np.maximum(z, 0); s = h @ Q["W2"] + Q["b2"]
    s -= s.max(1, keepdims=True); p = np.exp(s); p /= p.sum(1, keepdims=True); return p, (x, z, h)
def mlp_backward(Q, Y, cache, p):
    x, z, h = cache; ds = (p - Y) / len(Y); dh = ds @ Q["W2"].T * (z > 0)  # same chain rule, no patches
    return {"W2": h.T @ ds, "b2": ds.sum(0), "W1": x.T @ dh, "b1": dh.sum(0)}
Q = mlp_init(); r = np.random.default_rng(1); m = {k: 0 * v for k, v in Q.items()}; v2 = {k: 0 * v for k, v in Q.items()}; t = 0
for ep in range(15):
    for bi in np.array_split(r.permutation(len(Xtr)), len(Xtr) // 64):
        p, c = mlp_forward(Q, Xtr[bi]); g = mlp_backward(Q, Ytr[bi], c, p); t += 1
        for k in Q:
            m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
            Q[k] -= 0.01 * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)
print("MLP parameters:", sum(v.size for v in Q.values()))
print(f"CNN test accuracy {np.mean(forward(P, Xte)[0].argmax(1) == yte):.4f}   MLP test accuracy {np.mean(mlp_forward(Q, Xte)[0].argmax(1) == yte):.4f}")

<details><summary><b>Answer and explanation</b></summary>

On these tiny 8×8 images the two are about level (CNN 97.5%, MLP 98.1%); the gap is within run-to-run noise. The convolutional advantage is efficiency and structure, not magic: an MLP on a 224×224 colour image needs 150,000 weights per hidden unit, while a 3×3 conv filter needs 27 regardless of image size. At 8×8 the MLP isn't yet paying that price. Next step shows what the CNN's structure does buy, even here.

Expected output:
```
MLP parameters: 1510
CNN test accuracy 0.9750   MLP test accuracy 0.9806
```

</details>

## Milestone 7 · Shift the digits by one pixel

Shift every test image one pixel to the right (fill with zeros) and re-measure both models' accuracy. Which one drops more, and why?

In [ ]:
# Your turn
Xsh = np.zeros_like(Xte); Xsh[:, :, :, 1:] = ...  # TODO: move every image one pixel right
for name, f in [("CNN", lambda X: forward(P, X)[0]), ("MLP", lambda X: mlp_forward(Q, X)[0])]:
    a0, a1 = np.mean(f(Xte).argmax(1) == yte), np.mean(f(Xsh).argmax(1) == yte)
    print(f"{name}: original {a0:.4f}  shifted {a1:.4f}  drop {a0 - a1:.4f}")

In [ ]:
# @title Solution 7 (double-click to show the code)
Xsh = np.zeros_like(Xte); Xsh[:, :, :, 1:] = Xte[:, :, :, :-1]  # move every image one pixel right
for name, f in [("CNN", lambda X: forward(P, X)[0]), ("MLP", lambda X: mlp_forward(Q, X)[0])]:
    a0, a1 = np.mean(f(Xte).argmax(1) == yte), np.mean(f(Xsh).argmax(1) == yte)
    print(f"{name}: original {a0:.4f}  shifted {a1:.4f}  drop {a0 - a1:.4f}")

<details><summary><b>Answer and explanation</b></summary>

A one-pixel shift drops the CNN from 97.5% to 54.4% (−43 points) and the MLP from 98.1% to 46.1% (−52 points). The CNN is more robust because its filters detect a stroke wherever it is, but it still drops a lot. Pooling only absorbs shifts within its 2×2 windows, and the final dense layer is tied to positions again. On an 8×8 image one pixel is 12% of the width, which is huge. Real CNNs get much more shift-tolerance from many conv+pool stages and from training on shifted copies (data augmentation, D2L §14.1).

Expected output:
```
CNN: original 0.9750  shifted 0.5444  drop 0.4306
MLP: original 0.9806  shifted 0.4611  drop 0.5194
```

</details>